## Practical Homework 1 - Linear Regression

**Please write your name and student number in the following cell.**

In [ ]:
# Student_Name = "amirali moghadasi"
# Student_Number = "Amirali Moghadasi"

## Phase 0: Intro

For this assignment, you'll be given a dataset containing cars for sale. To study the dataset in more detail, refer to [this](https://www.kaggle.com/datasets/arianghasemi/divar-cars/data) link.

In [ ]:
import requests

url = "https://www.dropbox.com/scl/fi/cwd3e0wo7j41wungb4zat/cars.csv?rlkey=z83jkj9ex2no06z4htf6ce1w5&dl=1"
response = requests.get(url)

with open("dataset.csv", "wb") as f:
    f.write(response.content)

print("Downloaded successfully!")

In [ ]:
# Warning! You are not permitted to import any other libraries

import os
import pandas
import sklearn
import numpy as np
import pandas as pd

from tqdm import tqdm

import seaborn as sns
from joblib import dump, load
import seaborn as sns
from matplotlib import pyplot as plt

import warnings
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")

## Phase 1: Load Data (P1: 2 points)

Load the dataset (as a dataframe) using pandas and display the top 5 rows of the dataframe. **(P1-1: 1 points)**

In [ ]:
file_path = '/kaggle/working/dataset.csv'
df = pd.read_csv("dataset.csv")
df.head()

Drop any row containing null values. **(P1-2: 1 points)**

In [ ]:
df = df.dropna()
df.shape

## Phase 2: Preprocess (P2: 18 points)

Print names of the columns and the number of rows in the dataset. **(P2-1: 1 points)**

In [ ]:

print(" Column Names:\n")
for col in df.columns:
    print("-", col)

print("\n Number of rows:", df.shape[0])


Drop all rows where "Selling_Method" is not "نقدی", and then drop the columns "Selling_Method", "Time" and "Location". **(P2-2: 1 points)** 

In [ ]:

df = df[df["Selling_Method"] == "نقدی"]

df = df.drop(columns=["Selling_Method", "Time", "Location"])
df.shape


Retrieve the description of all columns in the dataset. **(P2-3: 1 points)**

In [ ]:
df.describe(include='all')


**We need to perform some operations on certain columns to make the entire dataset suitable for a Linear Regression task.**

Change the 'Model_Year' type to numerical. Convert all years written in the Gregorian format to the Solar format. Then, take 1365 as the start year and count the number of years passed since 1365. For example, the year 1399 must be substituted with 34, and the cars tagged with 'قبل از ۱۳۶۶' must be substituted with 0. **(P2-4: 3 points)**

In [ ]:
def change_to_solar(value):
    if "قبل" in str(value):
        return 0

    try:
        year = int(str(value).strip())
    except:
        return np.nan

    if year > 1500:
        year = year - 621

    if year >= 1365:
        return year - 1365
    else:
        return 0

df['Model_Year'] = df['Model_Year'].apply(change_to_solar)


Change the types of 'Mileage' and 'Third_party_insurance' to numerical. **(P2-5: 1 points)**

In [ ]:
# تبدیل اعداد فارسی به انگلیسی
def fa_to_en(text):
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    trans_table = str.maketrans(persian_digits, english_digits)
    return str(text).translate(trans_table)

def clean_numeric(col):
    return (
        col.astype(str)
        .apply(fa_to_en)
        .str.replace(",", "")
        .str.replace(r"[^0-9]", "", regex=True)
        .replace("", "0")
        .astype(float)
    )

df["Mileage"] = clean_numeric(df["Mileage"])
df["Third_party_insurance"] = clean_numeric(df["Third_party_insurance"])


For 'Engine_Condition', map 'سالم' to 2, 'تعویض شده' to 1, and 'نیاز به تعمیر' to 0. **(P2-6: 2 points)**

In [ ]:
ENGINE_WORKING = "سالم"
ENGINE_SUBSTITUTED = "تعویض شده"
ENGINE_NEEDS_TO_BE_FIXED = "نیاز به تعمیر"
engine_condition_map = {
    ENGINE_WORKING: 2,
    ENGINE_SUBSTITUTED: 1,
    ENGINE_NEEDS_TO_BE_FIXED: 0
}
df["Engine_Condition"] = df["Engine_Condition"].map(engine_condition_map)

print(df["Engine_Condition"])

For 'Chassis_Condition', map 'سالم و پلمپ' to 2, 'ضربه خورده' to 1, and 'رنگ شده' to 0. **(P2-7: 1 points)**

In [ ]:
CHASSIS_WORKING = "سالم و پلمپ"
CHASSIS_SUBSTITUTED = "ضربه خورده"
CHASSIS_NEEDS_TO_BE_FIXED = "رنگ شده"
chassis_condition_map = {
    CHASSIS_WORKING: 2,
    CHASSIS_SUBSTITUTED: 1,
    CHASSIS_NEEDS_TO_BE_FIXED: 0
}
df["Chassis_Condition"] = df["Chassis_Condition"].map(chassis_condition_map)
print(df["Chassis_Condition"])

For 'Gearbox', map 'دنده ای' to 0 and 'اتوماتیک' to 1. **(P2-8: 1 points)**

In [ ]:
GEARBOX_MANUAL = "دنده ای"
GEARBOX_AUTO = "اتوماتیک"
gearbox_map = {
    GEARBOX_AUTO: 1,
    GEARBOX_MANUAL: 0
}
df ["Gearbox"]=df ["Gearbox"].map(gearbox_map)
print(df["Gearbox"])

Convert the "City" column into one-hot format. Then drop the "City" column. You will need to create a column for each unique City, and in each row, the column corresponding to its City will be 1, and other columns will be 0. **(P2-9: 2 points)**

In [ ]:

city_dummies = pd.get_dummies(df["City"], prefix="City")
city_dummies = city_dummies.astype(int)
df = pd.concat([df, city_dummies], axis=1)

df = df.drop(columns=["City"])





Do the same steps for "Color" and "Body_Condition" columns **(P2-10: 2 points)**

In [ ]:
Color_dummies = pd.get_dummies(df["Color"], prefix="Color")
Color_dummies = Color_dummies.astype(int)

df = pd.concat([df, Color_dummies], axis=1)
df = df.drop(columns=["Color"])


In [ ]:
body_dummies = pd.get_dummies(df["Body_Condition"], prefix="Body")
body_dummies = body_dummies.astype(int)
df = pd.concat([df, body_dummies], axis=1)
df = df.drop(columns=["Body_Condition"])

Up until now, you have learned how to clean such a dataset. now, let's take other cars aside and focus on the "Pride". drop all the items in the dataframe where the "Brand" does not contain "پراید" and drop the "Brand" column **(P2-11: 3 points)**

In [ ]:
df = df[df["Brand"].str.contains("پراید", na=False)]
df = df.drop(columns=["Brand"])


## Phase 3: Exploration (P3: 10 points)

In [ ]:
df.head()

Create **a copy** of the first 7 columns of the dataframe (We won't visualize the one-hot features. so we don't need them). **(P3-1: 2 points)**

In [ ]:
df_small = df.iloc[:, :7].copy()

In [ ]:
df_small.head()

Utilize histograms and box plots to visualize the distribution of the  **(P3-2: 5 points)**

In [ ]:
# --- Histogram ---
df_small.hist(figsize=(12, 10))
plt.tight_layout()
plt.show()

# --- Boxplot ---
plt.figure(figsize=(12, 8))
df_small.boxplot()
plt.xticks(rotation=45)
plt.show()


Print the correlation coefficients matrix **(P3-3: 1 points)**

In [ ]:

corr_matrix = df_small.corr()
print(corr_matrix)


Plot `Price` plot according to`Mileage` using `seaborn.scatterplot` **(P3-4: 1 points)**

In [ ]:

sns.scatterplot(data=df_small, x="Mileage", y="Price")
plt.title("Price vs Mileage")
plt.show()



Plot `Model_Year` plot according to`Gearbox` using `seaborn.lineplot` **(P3-5: 1 points)**

In [ ]:
sns.lineplot(data=df_small, x="Gearbox", y="Model_Year")
plt.title("Model Year vs Gearbox")
plt.show()


It's clear that some columns are more correlated with the price. In practice, when dealing with many columns, we might set aside the least correlated ones and focus on the rest. However, here we will continue with all columns of 'df'.

## Phase 4: Training (P4: 55 points)

### Sec 1: Preparing features and Targets (P4-Sec1: 10 points)

Extract only the features from the dataframe by removing the target column ("Price").
Note: Do not remove the previous dataframe. **(P4-1-1: 2 points)**

In [ ]:

df_features = df.drop(columns=["Price"])
df_features.head()


Convert the new dataframe into a numpy array. **(P4-1-2: 1 points)**

In [ ]:
x = df_features.to_numpy()
print(x)

Get the target column from the previous dataframe and convert it to another numpy array named y **(P4-1-3: 1 points)**

In [ ]:
y = df["Price"].to_numpy()

Split the dataset into two parts such that the training set (denoted a`x_train` and `y_train`), contains 80% of the samples. **(P4-1-4: 3 points)**

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

Fit MinMaxScaler on the training data and apply it to the entire dataset to normalize all columns of data into the range [0-1]. **(P4-1-5: 4 points)**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(x_train)

x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)

### Sec 2: Linear Regression, Lasso and Ridge Regression from Scratch (P4-Sec2: 40 points)

In [ ]:
import numpy as np

class CustomLinearRegression:
    def __init__(self, regularization=None, alpha=1.0):
        '''
        Initialize the CustomLinearRegression object.

        Parameters:
        regularization (str or None): Type of regularization. Can be None, 'lasso', or 'ridge'.
        alpha (float): Regularization strength parameter.
        '''
        self.regularization = regularization
        self.alpha = alpha
        self.weights = None

    def fit(self, X, y):
        '''
        Fit the linear regression model to the training data.

        Parameters:
        X (array-like): Training input data.
        y (array-like): Target values.

        Returns:
        None
        '''
        # Add bias term to input data
        X = np.insert(X, 0, 1, axis=1)
        y = y.reshape((-1, 1))

        if self.regularization is None:

            # Solve for weights without regularization
           self.weights = np.linalg.pinv(X.T @ X) @ X.T @ y


        elif self.regularization == "lasso":

            # Solve for weights with Lasso regularization
            self.weights = self._lasso_regression(X, y)

        elif self.regularization == "ridge":

            # Solve for weights with Ridge regularization
            self.weights = self._ridge_regression(X, y)

        else:
            raise ValueError("Invalid regularization type. Please choose None, 'lasso', or 'ridge'.")

    def predict(self, X):
        '''
        Make predictions using the trained model.

        Parameters:
        X (array-like): Input data for prediction.

        Returns:
        array-like: Predicted target values.
        '''
        if self.weights is None:
            raise ValueError("Model has not been fitted yet. Please call the fit method first.")
        X = np.insert(X, 0, 1, axis=1)
        return (X @ self.weights).flatten()

    def _lasso_term(self, weights):
        '''
        Compute the Lasso regularization term.

        Parameters:
        weights (array-like): Model weights.

        Returns:
        float: Lasso regularization term.
        '''
        return self.alpha * np.sum(np.abs(weights[1:]))

    def _lasso_derivative(self, weights):
        '''
        Compute the derivative of the Lasso regularization term.

        Parameters:
        weights (array-like): Model weights.

        Returns:
        array-like: Derivative of the Lasso regularization term.
        '''
        grad = self.alpha * np.sign(weights)
        grad[0] = 0
        return grad

    def _lasso_regression(self, X, y):
        '''
        Perform Lasso regression to find model weights.

        Parameters:
        X (array-like): Input data.
        y (array-like): Target values.

        Returns:
        array-like: Model weights.
        '''
        X = X - X.mean()

        n,d = X.shape
        w = np.random.randn(d, 1)
        num_iterations = 500
        lr = 0.2

        for it in range(num_iterations):
            y_pred = np.dot(X, w)
            loss = (1 / (2*n)) * np.sum(np.square(y_pred - y)) + self._lasso_term(w)
            dw = (1/n) * np.dot(X.T, (y_pred - y)) + self._lasso_derivative(w)
            w = w - lr * dw
        return w

    def _ridge_regression(self, X, y):
        '''
        Perform Ridge regression to find model weights.

        Parameters:
        X (array-like): Input data.
        y (array-like): Target values.

        Returns:
        array-like: Model weights.
        '''
        n, d = X.shape
        I = np.eye(d)
        I[0, 0] = 0
        return np.linalg.inv(X.T @ X + self.alpha * I) @ X.T @ y


### Sec 3: Moddel Fitting (P4-Sec3: 5 points)

Fit 3 linear regressor to the data.
- Your implemented linear regression.
- Your implemented ridge.
- Your implemented lasso.

**(P4-3-1: 3 points)**

In [ ]:
my_regzrs = [
    ("Linear Regression", CustomLinearRegression()),
    ("Ridge", CustomLinearRegression(regularization="ridge")),
    ("Lasso", CustomLinearRegression(regularization="lasso"))
]
for name, model in my_regzrs:
    model.fit(x_train, y_train)
    print(name, "trained successfully")

Fit 3 linear regressor to the data.
- sklearn linear regression.
- sklearn lasso.
- sklearn ridge.

**(P4-3-2: 2 points)**

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge

sk_regzrs = [("Linear Regression", LinearRegression(fit_intercept=False)), ("Ridge", Ridge()), ("Lasso", Lasso())]
for name, model in sk_regzrs:
    model.fit(x_train, y_train)
    print(name, "trained successfully")


## Phase 5: Evaluation (P5: 15 points)

### Sec 1: RMSE (P5-Sec1: 10 points)

Calculate the root mean squared error of your three models on the `train` data. **(P5-1-1: 3 points)**

In [ ]:
from sklearn.metrics import mean_squared_error
for name, model in my_regzrs:
    y_pred = model.predict(x_train)
    rmse = np.sqrt(mean_squared_error(y_train, y_pred))
    print(name, "RMSE (train):", rmse)


Calculate the root mean squared error of sklearn's three models on the `train` data. **(P5-1-2: 2 points)**

In [ ]:
for name, model in sk_regzrs:
    y_pred = model.predict(x_train)
    rmse = np.sqrt(mean_squared_error(y_train, y_pred))
    print(name, "RMSE (train):", rmse)


Calculate the root mean squared error of your three models on the `test` data. **(P5-1-3: 3 points)**

In [ ]:
for name, model in my_regzrs:
    y_pred = model.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(name, "RMSE (test):", rmse)


Calculate the root mean squared error of sklearn's three models on the `test` data. **(P5-1-4: 2 points)**

In [ ]:
for name, model in sk_regzrs:
    y_pred = model.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(name, "RMSE (test):", rmse)


### Sec 2: Lasso (P5-Sec2: 5 points)

Get trained weights of scikit-learn's Lasso model. **(P5-2-1: 3 points)**

In [ ]:
weights = sk_regzrs[2][1].coef_


Sort and print the weights. **(P5-2-2: 2 points)**

In [ ]:
sorted_weights = np.sort(weights)
print(sorted_weights)
